# BloxLogicAI - Multivariate Tea-Export Forecasting (Research Notebook)

**Goal:** extend the univariate Prophet export-volume model into a **multivariate** one driven by
tea **production**, the **USD/LKR** exchange rate, and **weather** (rainfall, temperature), then decide
the final settings to port into the package (`utils/data_loader.py`, `models/forecasting.py`).

Run the cells top-to-bottom. The notebook reuses `utils.data_loader` where possible and prototypes the
new logic inline: production-weighted tea-region weather, Prophet regressors, cascade-forecasting of
future drivers, and scenario "what-if".

> Data note: all four drivers now span **2011-2026** (district-level weather + extended USD/LKR history),
> so the full multivariate model trains on the entire ~165-month export history. Printed windows reflect
> the actual coverage of whatever data is present.

In [ ]:
%matplotlib inline
import os, sys, glob, logging, warnings
warnings.filterwarnings("ignore")
for _n in ("prophet", "cmdstanpy"):
    logging.getLogger(_n).setLevel(logging.ERROR)

# locate repo root (walk up until requirements.txt is found)
ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "requirements.txt")) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import prophet as _p
from prophet import Prophet
from prophet.utilities import regressor_coefficients

from utils import data_loader as dl

pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (11, 4)
REG = ["production_mt", "usd_lkr_avg", "rainfall_mm", "temp_mean"]
print("repo root :", ROOT)
print("prophet   :", getattr(_p, "__version__", "?"))
print("regressors:", REG)

## 1 - Load the raw monthly series

Export and production come from the long SLTB customs files (2011-2026); USD/LKR FX from
`usd_lkr_historical.csv` (daily, 2011-2026); weather from the per-district daily files in
`data/sources/sri_lanka_weather_data/` (one CSV per district, 2011-2026).

In [ ]:
export = dl.load_export_monthly()           # month, export_mt     (long history)
prod_total = dl.load_production_monthly()    # month, production_mt  (long history)
# FX: USD/LKR daily (Yahoo 'LKR=X' export, 2011-2026); skip 2 metadata rows, date in 'Price', rate in 'Close'
fx_raw = pd.read_csv(os.path.join(ROOT, "data", "sources", "usd_lkr_historical.csv"), skiprows=[1, 2])
fx_raw = fx_raw.rename(columns={"Price": "date", "Close": "rate"})
fx_raw["date"] = pd.to_datetime(fx_raw["date"], errors="coerce")
fx_raw["rate"] = pd.to_numeric(fx_raw["rate"], errors="coerce")
fx_raw = fx_raw.dropna(subset=["date", "rate"])
fx_raw["month"] = fx_raw["date"].dt.to_period("M").dt.to_timestamp()
fx = (fx_raw.groupby("month")
            .agg(usd_lkr_avg=("rate", "mean"), usd_lkr_volatility=("rate", "std")).reset_index())

# district-by-district daily weather (2011-2026): one CSV per district
wx_files = sorted(glob.glob(os.path.join(ROOT, "data", "sources", "sri_lanka_weather_data", "*.csv")))
wx_raw = pd.concat((pd.read_csv(f) for f in wx_files), ignore_index=True)
wx_raw["date"] = pd.to_datetime(wx_raw["date"], errors="coerce")
wx_raw["temp_mean"] = (wx_raw["temperature_2m_max"] + wx_raw["temperature_2m_min"]) / 2.0
wx_raw = wx_raw.dropna(subset=["date"])

print("export     :", f"{export['month'].min():%Y-%m} -> {export['month'].max():%Y-%m}  ({len(export)} months)")
print("production :", f"{prod_total['month'].min():%Y-%m} -> {prod_total['month'].max():%Y-%m}  ({len(prod_total)} months)")
print("FX USD/LKR :", f"{fx['month'].min():%Y-%m} -> {fx['month'].max():%Y-%m}  ({len(fx)} months)")
print("weather    :", f"{wx_raw['date'].min():%Y-%m} -> {wx_raw['date'].max():%Y-%m}  ({wx_raw['district'].nunique()} districts)")

## 2 - Production by elevation zone -> monthly shares

The production file carries High / Medium / Low elevation rows per month (the shipped loader keeps only the
`Total`). Each zone's monthly share of national production is later used to weight the regional weather.

In [ ]:
pz = pd.read_csv(dl.PRODUCTION_CSV, usecols=range(4),
                 names=["month", "elev", "qty_kg", "mrec"], header=0)
pz["month"] = pd.to_datetime(pz["month"].fillna(pz["mrec"]), errors="coerce")
pz["qty_kg"] = pd.to_numeric(pz["qty_kg"], errors="coerce")
pz["elev"] = pz["elev"].astype(str).str.strip()
pz = pz[pz["elev"].isin(["High", "Medium", "Low"])].dropna(subset=["month", "qty_kg"])

zone_mt = (pz.groupby(["month", "elev"])["qty_kg"].sum().unstack("elev") / 1000.0)
zone_mt = zone_mt.reindex(columns=["High", "Medium", "Low"])
shares = zone_mt.div(zone_mt.sum(axis=1), axis=0)
print("zone production months:", f"{zone_mt.index.min():%Y-%m} -> {zone_mt.index.max():%Y-%m}")
print("avg production share  :", shares.mean().round(3).to_dict())

ax = shares.plot.area(ylim=(0, 1), color=["#1f3b2d", "#2e9e5b", "#9ed4b0"])
ax.set_title("Monthly tea production share by elevation zone")
ax.set_ylabel("share")
plt.tight_layout(); plt.show()

## 3 - Production-weighted tea-region weather

Average only the **tea-growing districts**, grouped by elevation zone, then combine the zones weighted by
each month's production share. Non-tea districts (Hambantota, Jaffna, Colombo, the dry zone, etc.) are
dropped. This district set includes **Nuwara Eliya** and **Badulla** - the prime high-grown districts.

In [ ]:
# Tea districts -> elevation zone (matches the High/Medium/Low production split).
DISTRICT_ZONES = {
    "Nuwara Eliya": "High", "Badulla": "High",          # high-grown
    "Kandy": "Medium", "Matale": "Medium",               # mid-grown
    "Ratnapura": "Low", "Kegalle": "Low", "Galle": "Low",
    "Matara": "Low", "Kalutara": "Low",                  # low-grown
}

def monthly_zone_weather(raw):
    w = raw[raw["district"].isin(DISTRICT_ZONES)].copy()
    w["zone"] = w["district"].map(DISTRICT_ZONES)
    w["month"] = w["date"].dt.to_period("M").dt.to_timestamp()
    # per district-month: total rain, mean temp -> per zone-month: mean across districts
    dm = (w.groupby(["zone", "district", "month"])
            .agg(rain=("rain_sum", "sum"), temp=("temp_mean", "mean")).reset_index())
    zm = (dm.groupby(["zone", "month"])
            .agg(rain=("rain", "mean"), temp=("temp", "mean")).reset_index())
    return (zm.pivot(index="month", columns="zone", values="rain"),
            zm.pivot(index="month", columns="zone", values="temp"))

def prod_weighted(zone_df, shares):
    cols = [c for c in ["High", "Medium", "Low"] if c in zone_df.columns]
    zone_df = zone_df[cols]
    w = shares.reindex(zone_df.index).reindex(columns=cols)
    w = w.fillna(pd.DataFrame(1.0, index=zone_df.index, columns=cols))  # equal-weight fallback
    w = w.where(zone_df.notna())                                        # ignore zones with no data
    w = w.div(w.sum(axis=1), axis=0)                                    # renormalise
    return (zone_df * w).sum(axis=1, min_count=1)

rain_z, temp_z = monthly_zone_weather(wx_raw)
weather = pd.DataFrame({
    "month": rain_z.index,
    "rainfall_mm": prod_weighted(rain_z, shares).values,
    "temp_mean": prod_weighted(temp_z, shares).values,
})

# compare with a naive all-district equal-weight average (incl. dry, non-tea districts)
naive = wx_raw.copy()
naive["month"] = naive["date"].dt.to_period("M").dt.to_timestamp()
naive_rain = (naive.groupby(["district", "month"])["rain_sum"].sum().groupby("month").mean())

fig, ax = plt.subplots()
ax.plot(naive_rain.index, naive_rain.values, label="naive all-district avg", color="#bbbbbb")
ax.plot(weather["month"], weather["rainfall_mm"], label="tea-zone, production-weighted", color="#2e9e5b")
ax.set_title("Monthly rainfall: tea-region weighted vs naive all-district average")
ax.set_ylabel("rainfall (mm)"); ax.legend()
plt.tight_layout(); plt.show()
print("weather months    :", f"{weather['month'].min():%Y-%m} -> {weather['month'].max():%Y-%m}")
print("tea districts used:", sorted(DISTRICT_ZONES))

## 4 - Assemble the multivariate frame

Left-join export (`y`) with the three drivers on month. The **all-driver window** is the set of months where
the target and every regressor are present. With all drivers now spanning 2011-2026 this is essentially the
full export history.

In [ ]:
df = (export.rename(columns={"month": "ds", "export_mt": "y"})
            .merge(prod_total.rename(columns={"month": "ds"}), on="ds", how="left")
            .merge(fx[["month", "usd_lkr_avg"]].rename(columns={"month": "ds"}), on="ds", how="left")
            .merge(weather.rename(columns={"month": "ds"}), on="ds", how="left")
            .sort_values("ds").reset_index(drop=True))

trainable = df.dropna(subset=["y"] + REG).reset_index(drop=True)
pw = df.dropna(subset=["y", "production_mt", "rainfall_mm", "temp_mean"])
print("export history     :", f"{df['ds'].min():%Y-%m} -> {df['ds'].max():%Y-%m}  ({int(df['y'].notna().sum())} months)")
print("prod+weather window:", f"{pw['ds'].min():%Y-%m} -> {pw['ds'].max():%Y-%m}  ({len(pw)} months)")
print("all-driver window  :", f"{trainable['ds'].min():%Y-%m} -> {trainable['ds'].max():%Y-%m}  ({len(trainable)} months)")

z = trainable.set_index("ds")[["y"] + REG]
zc = (z - z.mean()) / z.std()
ax = zc.plot(color=["#1f3b2d", "#2e9e5b", "#d98e04", "#3b78c2", "#9b4dca"])
ax.set_title("Export vs drivers (z-scored) over the all-driver window")
ax.set_ylabel("z-score")
plt.tight_layout(); plt.show()

print("\nlagged correlation vs export (driver leads export by k months):")
for col in REG:
    cors = {k: round(pw["y"].corr(pw[col].shift(k)), 2) for k in range(4)}
    print(f"  {col:14s}", cors)

## 5 - Modelling helpers

`fit_prophet` (Prophet + optional additive regressors), `forecast_regressor` (cascade: project one driver
with its own Prophet), and `backtest` (hold out the last N months and score on an exact, gap-safe frame).

In [ ]:
PARAMS = dict(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
              seasonality_mode="multiplicative", changepoint_prior_scale=0.5, interval_width=0.90)

def fit_prophet(train, regressors=()):
    m = Prophet(**PARAMS)
    for r in regressors:
        m.add_regressor(r, mode="additive")
    m.fit(train[["ds", "y"] + list(regressors)])
    return m

def mape(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    return float(np.mean(np.abs((actual - pred) / actual)) * 100)

def forecast_regressor(train, col, future_ds):
    s = train[["ds", col]].rename(columns={col: "y"}).dropna()
    mr = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    mr.fit(s)
    return mr.predict(pd.DataFrame({"ds": list(future_ds)}))["yhat"].to_numpy()

def backtest(frame, regressors=(), test_n=12, future="actual"):
    regressors = list(regressors)
    f = frame.dropna(subset=["y"] + regressors).sort_values("ds").reset_index(drop=True)
    tr, te = f.iloc[:-test_n], f.iloc[-test_n:]
    m = fit_prophet(tr, regressors)
    if not regressors:
        fut = pd.concat([tr[["ds"]], te[["ds"]]])
    elif future == "forecast":
        fr = te[["ds"]].copy()
        for r in regressors:                       # drivers projected from TRAIN only (no leakage)
            fr[r] = forecast_regressor(tr, r, te["ds"])
        fut = pd.concat([tr[["ds"] + regressors], fr[["ds"] + regressors]])
    else:                                          # oracle: real held-out drivers
        fut = pd.concat([tr[["ds"] + regressors], te[["ds"] + regressors]])
    pred = m.predict(fut).set_index("ds").reindex(te["ds"])["yhat"].to_numpy()
    return {"mape": round(mape(te["y"], pred), 2),
            "mae": round(float(np.mean(np.abs(te["y"].to_numpy() - pred))), 1),
            "n_train": int(len(tr)),
            "window": f"{f['ds'].min():%Y-%m}..{f['ds'].max():%Y-%m}"}

print("helpers ready")

## 6 - Univariate baseline

Reproduce the shipped model: Prophet on export history alone. Reported both on the full continuous history
and restricted to the all-driver window (for a like-for-like comparison with the multivariate model).

In [ ]:
uni_full = dl.build_forecast_dataset()[["ds", "y"]]    # full continuous (interpolated) history
print("univariate (full history)     :", backtest(uni_full, (), 12))
print("univariate (all-driver window):", backtest(trainable[["ds", "y"]], (), 12))

## 7 - Multivariate Prophet and driver effects

Fit Prophet with the three drivers as additive regressors and inspect the learned coefficients
(on standardised drivers): sign and magnitude show how each driver moves exports.

In [ ]:
m_mv = fit_prophet(trainable, REG)
coef = regressor_coefficients(m_mv)
print("Learned regressor effects (additive, standardised drivers):")
display(coef)

print("\nbacktests on the all-driver window:")
print("  multivariate (oracle drivers)  :", backtest(trainable, REG, 12, future="actual"))
print("  multivariate (forecast drivers):", backtest(trainable, REG, 12, future="forecast"))

## 8 - Future drivers: auto-estimate + what-if scenario

To forecast ahead, the drivers' future values are needed. Default = cascade auto-estimate (each driver
projected by its own Prophet). A what-if scenario then overrides them (here: FX shock to 400 LKR/USD and a
10% production drop) - the basis of the dashboard's scenario sliders.

In [ ]:
last = trainable["ds"].max()
future_ds = pd.date_range(last + pd.offsets.MonthBegin(1), periods=12, freq="MS")
fut_base = pd.DataFrame({"ds": future_ds})
for r in REG:
    fut_base[r] = forecast_regressor(trainable, r, future_ds)

fut_shock = fut_base.copy()
fut_shock["usd_lkr_avg"] = 400.0
fut_shock["production_mt"] = fut_shock["production_mt"] * 0.90

def project(model, hist, fut):
    frame = pd.concat([hist[["ds"] + REG], fut[["ds"] + REG]])
    return model.predict(frame).set_index("ds").loc[fut["ds"], "yhat"]

base = project(m_mv, trainable, fut_base)
shock = project(m_mv, trainable, fut_shock)

fig, ax = plt.subplots()
tail = trainable.set_index("ds")["y"].iloc[-24:]
ax.plot(tail.index, tail.values, label="history", color="#1f3b2d")
ax.plot(base.index, base.values, "--", label="baseline (auto-estimated drivers)", color="#2e9e5b")
ax.plot(shock.index, shock.values, "--", label="what-if: FX=400, production -10%", color="#d62728")
ax.set_title("12-month export forecast: baseline vs what-if scenario")
ax.set_ylabel("export (MT)"); ax.legend()
plt.tight_layout(); plt.show()
print("baseline next-month:", round(float(base.iloc[0]), 0), "MT")
print("scenario next-month:", round(float(shock.iloc[0]), 0), "MT")

## 9 - Backtest comparison and regressor-subset sweep

Each subset is scored on its **own maximal window** (see the `window` column); with all drivers now spanning
2011-2026 these are essentially the full history. Compared against the univariate baseline, with oracle
drivers (ceiling) and forecast drivers (operational / real-world).

In [ ]:
rows = []
rows.append(("univariate (full history)", backtest(uni_full, (), 12)))
rows.append(("univariate (window)", backtest(trainable[["ds", "y"]], (), 12)))

def short(s):
    return (s.replace("production_mt", "prod").replace("usd_lkr_avg", "fx")
             .replace("rainfall_mm", "rain").replace("temp_mean", "temp"))

for subset in [["production_mt"],
               ["production_mt", "rainfall_mm", "temp_mean"],
               ["production_mt", "usd_lkr_avg"],
               REG]:
    name = "mv: " + "+".join(short(s) for s in subset)
    rows.append((name + " [oracle]", backtest(df, subset, 12, future="actual")))
    rows.append((name + " [forecast]", backtest(df, subset, 12, future="forecast")))

table = pd.DataFrame([dict(model=n, **r) for n, r in rows]).set_index("model").sort_values("mape")
display(table)

## 10 - Conclusions -> settings to port into the package

Read the table above (sorted by MAPE) and record the winning configuration; these are the inputs to
**Phase B** (the modular refactor):

- **Regressor set** - the subset with the best *forecast* (operational) MAPE that still matches or beats the
  univariate baseline on a comparable window. Production+weather now train on the full history; FX is still
  capped at 2019 until that source is extended.
- **Regressor mode** - additive (used here) vs multiplicative: try both in `fit_prophet`.
- **Lags** - if section 4 shows a driver leads exports, add a lagged column in the pipeline.
- **Weather** - the district->zone map in section 3 plus production-weighting; confirm the tea-zone signal is
  cleaner than the naive average.
- **Future drivers** - cascade auto-estimate (section 8) as default, with what-if overrides in the dashboard.

> Headline metric for the dissertation = **multivariate (forecast drivers)** MAPE vs the univariate baseline
> on the **same window**. The oracle column shows the ceiling if the drivers were known exactly.